# 🔍 Google Image Finder Agent (via Serper.dev)

Finds image URLs from **Google Image Search** using the [Serper.dev](https://serper.dev) API.

**Why Serper?** Google's own Custom Search API is paid + limited. Serper gives real Google results with a free tier (2,500 queries).

**Setup (1 min):**
1. Go to https://serper.dev and sign up (free)
2. Copy your API key from the dashboard
3. Paste it below when prompted

**Bonus:** Includes a Bing fallback that needs NO key.

In [1]:
# STEP 1: Install deps
%pip install requests ipython --quiet

/Users/somnathdas/Projects/tutorial-agentic-ai/server/.venv/bin/python: No module named pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
# STEP 3: Imports
import requests
from IPython.display import display, HTML
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
# STEP 4a: Google Image Search (via Serper)
def search_google_images(topic, max_results=8):
    """Real Google Image Search results via Serper.dev."""
    if not os.environ.get("SERPER_API_KEY"):
        print("❌ Add your Serper API key in Step 2 first.")
        return []

    url = "https://google.serper.dev/images"
    payload = {"q": topic, "num": max_results}
    headers = {"X-API-KEY": os.environ.get("SERPER_API_KEY"), "Content-Type": "application/json"}

    try:
        resp = requests.post(url, json=payload, headers=headers, timeout=20)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"❌ Network error: {e}")
        return []

    data = resp.json()
    images = data.get("images", [])
    if not images:
        print(f"😕 No results for '{topic}'")
        return []

    return [
        {
            "title":  img.get("title", ""),
            "url":    img.get("imageUrl"),      # full-size image
            "thumb":  img.get("thumbnailUrl"),   # preview
            "source": img.get("source", ""),     # website it came from
            "width":  img.get("imageWidth"),
            "height": img.get("imageHeight"),
        }
        for img in images[:max_results]
    ]

print("✅ Google (Serper) function loaded")

✅ Google (Serper) function loaded


In [4]:
# STEP 4b: Bing Image Search (no key needed, free)
def search_bing_images(topic, max_results=8):
    """Free Bing image search — no API key required."""
    url = "https://www.bing.com/images/async"
    params = {"q": topic, "first": str(max_results), "count": str(max_results)}
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 "
                      "(KHTML, like Gecko) Chrome/120.0 Safari/537.36"
    }

    try:
        resp = requests.get(url, params=params, headers=headers, timeout=20)
        resp.raise_for_status()
    except requests.RequestException as e:
        print(f"❌ Network error: {e}")
        return []

    # Bing returns a JSON blob embedded in <script> tags. Extract it.
    import re, json
    matches = re.findall(r'm\s*=\s*"(\{.*?\})"', resp.text)
    if not matches:
        print(f"😕 No results for '{topic}'")
        return []

    try:
        data = json.loads(matches[0].encode().decode("unicode_escape"))
    except Exception:
        print("⚠️  Could not parse Bing response, Google mode recommended")
        return []

    results = []
    for item in data.get("items", [])[:max_results]:
        media = item.get("media", {})
        results.append({
            "title":  item.get("title", ""),
            "url":    media.get("m"),
            "thumb":  media.get("m"),
            "source": item.get("desc", ""),
            "width":  None,
            "height": None,
        })
    return results

print("✅ Bing (no-key) function loaded")

✅ Bing (no-key) function loaded


In [5]:
# STEP 5: Pretty display
def show_results(results, topic):
    if not results:
        return
    print(f"\n🔎 {len(results)} images for: '{topic}'\n")
    html = '<div style="display:grid;grid-template-columns:repeat(2,1fr);gap:12px;">'
    for r in results:
        size = f"{r['width']}×{r['height']}" if r['width'] else ""
        html += f'''
        <div style="border:1px solid #ddd;border-radius:8px;padding:10px;font-family:sans-serif;font-size:12px;">
          <a href="{r['url']}" target="_blank">
            <img src="{r['thumb']}" style="width:100%;border-radius:6px;display:block;" />
          </a>
          <b style="display:block;margin-top:6px;">{r['title'][:60]}</b>
          <span style="color:#666;">🌐 {r['source'][:40]} {('• 📐 ' + size) if size else ''}</span><br/>
          <a href="{r['url']}" target="_blank" style="color:#0a66c2;">Open full image →</a>
        </div>'''
    html += "</div>"
    display(HTML(html))

print("✅ Display helper loaded")

✅ Display helper loaded


In [8]:
# STEP 6: 🔥 Run the agent
topic = "small language 2026"   # 👈 change this
engine = "google"           # "google" (needs key) or "bing" (no key)

if engine == "google":
    results = search_google_images(topic, max_results=8)
else:
    results = search_bing_images(topic, max_results=8)

show_results(results, topic)


🔎 8 images for: 'small language 2026'



In [10]:
# STEP 7 (optional): Get raw URL list
for i, r in enumerate(results, 1):
    print(f"{i}. {r['url']}")

1. https://mymodernmet.com/wp/wp-content/uploads/2020/10/cooper-baby-corgi-dogs-8.jpg
2. https://t3.ftcdn.net/jpg/02/74/06/48/360_F_274064877_Tuq84kGOn5nhyIJeUFTUSvXaSeedAOTT.jpg
3. https://i.pinimg.com/564x/22/fd/1d/22fd1db541a0074b0ea3176814fddf60.jpg
4. https://i.pinimg.com/736x/44/02/b6/4402b610a04ca5797b426faa9ea9a7e6.jpg
5. https://i.ytimg.com/vi/XDGMxC0GH1Y/maxresdefault.jpg
6. https://i.ytimg.com/vi/BTZJA3kMqOc/hq720.jpg?sqp=-oaymwEhCK4FEIIDSFryq4qpAxMIARUAAAAAGAElAADIQj0AgKJD&rs=AOn4CLA-IAEnOsa45KCoEefIlotig-5eoQ
7. https://preview.redd.it/corgi-puppies-are-so-cute-they-are-definitely-my-favorites-v0-mpg2sf6bfz271.jpg?width=960&format=pjpg&auto=webp&s=cc77756c5997b03ef4eb3004364ea86ea4cae15f
8. https://pethelpful.com/.image/NDowMDAwMDAwMDAwMTA5OTM4/a-corgi-puppy-running.jpg?profile=share4-3&x=37&y=48
